In [ ]:
import os
import re
import csv
from collections import defaultdict
import matplotlib.pyplot as plt
from typing import Dict, List

In [ ]:
def parse_result_file(file_path: str) -> str:
    with open(file_path, 'r') as f:
        content = f.read().strip()
    if content == 's SATISFIABLE':
        return 'SAT'
    elif content == 's UNSATISFIABLE':
        return 'UNSAT'
    elif content == 'c UNKNOWN':
        return 'TIMEOUT'
    else:
        return 'ERROR'

def parse_stderr_file(file_path: str) -> float:
    with open(file_path, 'r') as f:
        content = f.read()
    match = re.search(r'real\s+(\d+)m(\d+\.\d+)s', content)
    if match:
        minutes, seconds = match.groups()
        return float(minutes) * 60 + float(seconds)
    return None

In [ ]:
def analyze_results(base_dir: str) -> Dict[str, Dict[str, str | float]]:
    results = defaultdict(lambda: {'result': None, 'time': None})
    
    results_dir = os.path.join(base_dir, 'results')
    stderr_dir = os.path.join(base_dir, 'stderr')
    
    for filename in os.listdir(results_dir):
        if filename.endswith('_result.txt'):
            cnf_name = filename.replace('_result.txt', '')
            result_path = os.path.join(results_dir, filename)
            stderr_path = os.path.join(stderr_dir, f"{cnf_name}_stderr.txt")
            
            result = parse_result_file(result_path)
            time = parse_stderr_file(stderr_path) if os.path.exists(stderr_path) else None
            
            results[cnf_name] = {'result': result, 'time': time}
    
    return results


In [ ]:

def compare_results(baseline_results: Dict[str, Dict[str, str | float]], 
                    other_results: Dict[str, Dict[str, Dict[str, str | float]]]) -> List[Dict]:
    comparison = []
    
    for cnf_name in set(baseline_results.keys()) | set.union(*[set(res.keys()) for res in other_results.values()]):
        row = {
            'cnf_name': cnf_name,
            'baseline_result': baseline_results[cnf_name]['result'],
            'baseline_time': baseline_results[cnf_name]['time'],
        }
        
        for algo_name, results in other_results.items():
            row[f'{algo_name}_result'] = results[cnf_name]['result'] if cnf_name in results else 'N/A'
            row[f'{algo_name}_time'] = results[cnf_name]['time'] if cnf_name in results else None
            
            # Validate results
            if row[f'{algo_name}_result'] == row['baseline_result']:
                row[f'{algo_name}_validity'] = 'VALID'
            elif row[f'{algo_name}_result'] == 'TIMEOUT':
                row[f'{algo_name}_validity'] = 'TIMEOUT'
            elif row[f'{algo_name}_result'] == 'N/A':
                row[f'{algo_name}_validity'] = 'N/A'
            else:
                row[f'{algo_name}_validity'] = 'INVALID'
            
            # Calculate speedup
            if row['baseline_time'] is not None and row[f'{algo_name}_time'] is not None:
                if row[f'{algo_name}_result'] == 'TIMEOUT':
                    row[f'{algo_name}_speedup'] = 'TIMEOUT'
                else:
                    row[f'{algo_name}_speedup'] = row['baseline_time'] / row[f'{algo_name}_time']
            else:
                row[f'{algo_name}_speedup'] = None
        
        comparison.append(row)
    
    return comparison

In [ ]:
def write_comparison_csv(comparison: List[Dict], output_file: str, algo_names: List[str]):
    fieldnames = ['cnf_name', 'baseline_result', 'baseline_time']
    for algo in algo_names:
        fieldnames.extend([f'{algo}_result', f'{algo}_time', f'{algo}_validity', f'{algo}_speedup'])
    
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in comparison:
            writer.writerow(row)
    
    print(f"Comparison results have been written to {output_file}")

In [ ]:
def plot_results(comparison: List[Dict], output_dir: str, algo_names: List[str]):
    # Prepare data
    validity_counts = {algo: {'VALID': 0, 'INVALID': 0, 'TIMEOUT': 0, 'N/A': 0} for algo in algo_names}
    speedups = {algo: [] for algo in algo_names}
    
    for row in comparison:
        for algo in algo_names:
            validity_counts[algo][row[f'{algo}_validity']] += 1
            if row[f'{algo}_speedup'] not in (None, 'TIMEOUT'):
                speedups[algo].append(row[f'{algo}_speedup'])
    
    # Speedup distribution
    plt.figure(figsize=(12, 6))
    for algo in algo_names:
        plt.hist(speedups[algo], label=algo, alpha=0.7, bins=20)
    plt.xlabel('Speedup (Baseline Time / Algorithm Time)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Speedup')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'speedup_distribution.png'))
    plt.close()

    # Validity pie charts
    for algo in algo_names:
        plt.figure(figsize=(8, 8))
        labels = [k for k, v in validity_counts[algo].items() if v > 0]
        sizes = [v for k, v in validity_counts[algo].items() if v > 0]
        plt.pie(sizes, labels=labels, autopct='%1.1f%%')
        plt.title(f'Validity of {algo} Results')
        plt.savefig(os.path.join(output_dir, f'validity_pie_chart_{algo}.png'))
        plt.close()

    # Scatter plot of execution times
    for algo in algo_names:
        times_baseline = []
        times_algo = []
        for row in comparison:
            if (row['baseline_time'] is not None and 
                row[f'{algo}_time'] is not None and 
                row[f'{algo}_result'] != 'TIMEOUT'):
                times_baseline.append(row['baseline_time'])
                times_algo.append(row[f'{algo}_time'])
        
        plt.figure(figsize=(10, 6))
        plt.scatter(times_baseline, times_algo, alpha=0.5)
        plt.xlabel('Baseline Execution Time (s)')
        plt.ylabel(f'{algo} Execution Time (s)')
        plt.title(f'Execution Time Comparison: Baseline vs {algo} (Excluding Timeouts)')
        max_time = max(max(times_baseline), max(times_algo))
        plt.plot([0, max_time], [0, max_time], 'r--')  # Add y=x line
        plt.savefig(os.path.join(output_dir, f'execution_time_scatter_{algo}.png'))
        plt.close()
    
    print(f"Plots have been saved in {output_dir}")

In [ ]:
# Specify your directories here
baseline_dir = "/path/to/baseline/results"
other_algo_dirs = {
    "Algorithm1": "/path/to/algorithm1/results",
    "Algorithm2": "/path/to/algorithm2/results",
    # Add more algorithms as needed
}
output_dir = "/path/to/output/directory"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Analyze results
baseline_results = analyze_results(baseline_dir)
other_results = {name: analyze_results(dir) for name, dir in other_algo_dirs.items()}

# Compare results
comparison = compare_results(baseline_results, other_results)

# Write CSV
csv_file = os.path.join(output_dir, 'comparison_results.csv')
write_comparison_csv(comparison, csv_file, list(other_algo_dirs.keys()))

# Plot results
plot_results(comparison, output_dir, list(other_algo_dirs.keys()))

print("Analysis complete!")